# Module 2 — OpenSearch Serverless: the same retrieval, in production shape

Now we load the **same documents with the same Titan vectors** into an OpenSearch Serverless **NextGen** vector search collection and run the same k-NN query we ran in Chroma.

Before running this notebook:

1. `infra/create_collection.sh` finished and printed `Collection is ACTIVE`.
2. `.env` has `OPENSEARCH_HOST` (without `https://`) and `AWS_REGION=us-east-1`.
3. Module 1 ran, so the Titan vectors are already cached in `data/cache/`.

> **Scale to zero.** Our collection group has min OCU = 0. After about 10 minutes without traffic, compute scales to zero, and the first request afterwards can take 10-30 s while capacity comes back. That is expected, not a bug.

In [ ]:
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from pprint import pprint

from opensearchpy import helpers
from opensearchpy.exceptions import RequestError

from src.common import (
    EMBEDDING_DIM,
    INDEX_NAME,
    TOP_K,
    get_opensearch_client,
    load_docs,
    load_queries,
    recall_at_k,
)
from src.embeddings import get_embedder

docs = load_docs()
queries = load_queries()
embedder = get_embedder()
print("embedding provider:", embedder.name, "| dimension:", embedder.dimension)
if embedder.dimension != EMBEDDING_DIM:
    print(f"WARNING: expected {EMBEDDING_DIM} dims (Titan V2). Results will not match the article.")

## Connect with SigV4

OpenSearch Serverless has no usernames or passwords. Every HTTP request is signed with **AWS Signature Version 4** using our IAM credentials, and the service checks two things:

1. **IAM**: our principal can call `aoss:APIAccessAll`.
2. **Data-access policy**: the collection's policy lists our principal with the index/collection permissions we need.

`get_opensearch_client()` in `src/common.py` builds an `AWS4Auth` signer with service name **`aoss`** (OpenSearch *Serverless*; managed domains use `es`). Credentials come from the default boto3 chain, so nothing secret lives in the repo.

The first call below may be slow if the collection scaled to zero.

In [ ]:
client = get_opensearch_client()

start = time.perf_counter()
exists = client.indices.exists(index=INDEX_NAME)
print(f"index '{INDEX_NAME}' exists: {exists}  (first request took {time.perf_counter() - start:.1f} s)")

## The index mapping, field by field

In Chroma, a collection only needed a name. In OpenSearch we declare the **schema** (the *mapping*) up front, and it decides what each field can do:

| Field | Type | Why |
|---|---|---|
| `id` | `keyword` | Exact value, not analyzed. Lets us filter or look up by doc id. We also use it as the document `_id`. |
| `text` | `text` | Analyzed into terms (lowercased, split on spaces and punctuation) for **BM25** full-text search. This is what Chroma does not give us. |
| `lang`, `topic` | `keyword` | Metadata for exact filters, like Chroma's `metadatas`. |
| `embedding` | `knn_vector` | The vector field used by k-NN search. |
| `embedding.dimension` | `1024` | Must equal the length of every vector we send (Titan V2 = 1024). A mismatch is rejected at index time. |
| `embedding.method.name` | `hnsw` | The graph-based approximate nearest-neighbor algorithm (the same family Chroma uses). |
| `embedding.method.engine` | `faiss` | The library that builds and searches the HNSW graph. |
| `embedding.method.space_type` | `cosinesimil` | Cosine similarity, matching the `cosine` space we used in Chroma. |

`settings.index.knn: true` turns on the k-NN plugin for this index.

**NextGen note.** The AWS docs say NextGen vector collections *do not require* `engine` and `mode` because the service picks the configuration internally, and the console says the engine option is only configurable for Classic collections. We still try the explicit mapping first, because it is what we would write on a self-managed cluster. If the service rejects it, the next cell prints the error and retries without `engine`, so the notebook keeps running and we can see exactly what NextGen accepts.

In [ ]:
def build_mapping(dimension: int, explicit_engine: bool) -> dict:
    method = {"name": "hnsw", "space_type": "cosinesimil"}
    if explicit_engine:
        method["engine"] = "faiss"
    return {
        "settings": {"index": {"knn": True}},
        "mappings": {
            "properties": {
                "id": {"type": "keyword"},
                "text": {"type": "text"},
                "lang": {"type": "keyword"},
                "topic": {"type": "keyword"},
                "embedding": {"type": "knn_vector", "dimension": dimension, "method": method},
            }
        },
    }


if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)
    print("deleted existing index")

try:
    client.indices.create(index=INDEX_NAME, body=build_mapping(embedder.dimension, explicit_engine=True))
    print("created index with explicit hnsw + faiss mapping")
except RequestError as exc:
    print("explicit engine rejected:", exc.status_code, exc.error, exc.info)
    client.indices.create(index=INDEX_NAME, body=build_mapping(embedder.dimension, explicit_engine=False))
    print("created index with the NextGen simplified mapping (no engine)")

pprint(client.indices.get_mapping(index=INDEX_NAME)[INDEX_NAME]["mappings"]["properties"]["embedding"])

## Index the documents (same vectors as Chroma)

We send all documents in one `_bulk` request. Each document carries its text, metadata and its Titan vector. We set `_id` to our doc id so results line up with Chroma (NextGen collections support custom document ids).

OpenSearch Serverless does not support `refresh=true` on writes, and NextGen vector collections have a read-after-write delay of about 10 s. So we poll the document count instead of searching immediately.

In [ ]:
doc_vectors = embedder.embed([d["text"] for d in docs])

actions = [
    {
        "_index": INDEX_NAME,
        "_id": d["id"],
        "_source": {"id": d["id"], "text": d["text"], "lang": d["lang"], "topic": d["topic"], "embedding": vec},
    }
    for d, vec in zip(docs, doc_vectors)
]
ok, errors = helpers.bulk(client, actions, raise_on_error=False)
print(f"indexed: {ok}, errors: {errors}")

In [ ]:
for attempt in range(30):
    count = client.count(index=INDEX_NAME)["count"]
    print(f"searchable docs: {count}/{len(docs)}")
    if count == len(docs):
        break
    time.sleep(5)
else:
    raise RuntimeError("Documents are not searchable yet; wait a bit and re-run this cell.")

## k-NN query

The `knn` query takes our query vector and `k`, the number of neighbors to find. `size` is how many hits to return. We exclude `_source.embedding` to keep the response small.

In [ ]:
query = queries[5]
query_vector = embedder.embed([query["text"]])[0]

body = {
    "size": TOP_K,
    "_source": {"excludes": ["embedding"]},
    "query": {"knn": {"embedding": {"vector": query_vector, "k": TOP_K}}},
}
response = client.search(index=INDEX_NAME, body=body)

print("query:", query["text"])
for hit in response["hits"]["hits"]:
    print(f"  {hit['_id']}  score={hit['_score']:.4f}  {hit['_source']['text'][:70]}...")

### All queries, and a check against Chroma

We run every query and build the same `results` dict as in Module 1. If `chroma_db/` exists, we also check how many top-3 ids match Chroma. With identical vectors and the same metric, differences can only come from the approximate (HNSW) search and from how each engine breaks ties.

In [ ]:
query_vectors = embedder.embed([q["text"] for q in queries])

results = {}
for q, vec in zip(queries, query_vectors):
    body = {"size": TOP_K, "_source": False, "query": {"knn": {"embedding": {"vector": vec, "k": TOP_K}}}}
    ids = [h["_id"] for h in client.search(index=INDEX_NAME, body=body)["hits"]["hits"]]
    results[q["id"]] = {"kind": q["kind"], "retrieved": ids, "relevant": q["relevant_ids"],
                        "recall@3": recall_at_k(ids, q["relevant_ids"])}

pprint(results, sort_dicts=False, width=110)
for kind in ("keyword", "semantic", "mixed"):
    scores = [r["recall@3"] for r in results.values() if r["kind"] == kind]
    print(f"mean recall@3 [{kind:<8}] = {sum(scores) / len(scores):.2f}  (n={len(scores)})")

In [ ]:
from src.common import CHROMA_COLLECTION, CHROMA_PATH

if CHROMA_PATH.exists():
    import chromadb

    chroma = chromadb.PersistentClient(path=str(CHROMA_PATH)).get_collection(CHROMA_COLLECTION, embedding_function=None)
    chroma_ids = chroma.query(query_embeddings=query_vectors, n_results=TOP_K)["ids"]
    same = sum(results[q["id"]]["retrieved"] == ids for q, ids in zip(queries, chroma_ids))
    print(f"queries with identical top-{TOP_K} order in Chroma and OpenSearch: {same}/{len(queries)}")
else:
    print("chroma_db/ not found; run 01_chroma_setup.ipynb to compare.")

> **What you learned**
> - OpenSearch Serverless authenticates every request with SigV4 (service `aoss`); access needs both IAM and a data-access policy.
> - We declare a mapping up front: `text` for BM25, `keyword` for exact filters, `knn_vector` (dimension 1024) for vectors.
> - NextGen vector collections choose the k-NN engine internally; `engine` is optional there.
> - Writes become searchable after a short delay (no `refresh=true` on Serverless), so we poll before querying.
> - With identical vectors and metric, OpenSearch k-NN returns essentially the same neighbors as Chroma.

Next: `03_hybrid_search.ipynb`.